# PCA on Wine Dataset

Notebook này chuyển toàn bộ phân tích PCA từ file Python sang Jupyter để chạy từng phần riêng lẻ.

## 1. Chuẩn bị thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

## 2. Tải dữ liệu

In [ ]:
df = pd.read_csv("Wine dataset.csv")
df.columns = df.columns.str.strip()

print("===== FIRST 5 ROWS =====")
print(df.head())

print("\n===== DATASET INFORMATION =====")
print(df.info())

print("\n===== STATISTICAL SUMMARY =====")
print(df.describe())

## 3. Làm sạch dữ liệu

In [ ]:
print("\n===== MISSING VALUES =====")
print(df.isnull().sum())

print("\n===== DUPLICATED ROWS =====")
print("Number of duplicated rows:", df.duplicated().sum())

df = df.drop_duplicates()

print("\nDataset shape after cleaning:", df.shape)

## 4. Tách features và label

In [ ]:
X = df.drop("class", axis=1)
y = df["class"]

print("\n===== PCA INPUT FEATURES =====")
print(X.columns.tolist())

print("\nNumber of samples:", X.shape[0])
print("Number of original features:", X.shape[1])

## 5. Chia train/test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n===== TRAIN / TEST SPLIT =====")
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print("\nOriginal class distribution:")
print(y.value_counts(normalize=True))

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True))

## 6. Chuẩn hóa dữ liệu

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n===== STANDARDIZATION COMPLETED =====")
print("Mean of scaled training data:", np.round(X_train_scaled.mean(), 4))
print("Standard deviation of scaled training data:", np.round(X_train_scaled.std(), 4))

## 7. Baseline classifier

In [ ]:
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_scaled, y_train)

y_pred_baseline = baseline_model.predict(X_test_scaled)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print("\n===== BASELINE CLASSIFIER RESULT =====")
print("Model: Logistic Regression")
print("Input: Original 13 standardized features")
print("Baseline accuracy:", baseline_accuracy)

print("\nClassification report:")
print(classification_report(y_test, y_pred_baseline))

## 8. Áp dụng PCA với 2 thành phần

In [ ]:
pca = PCA(n_components=2)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("\n===== PCA RESULT =====")
print("Training PCA shape:", X_train_pca.shape)
print("Testing PCA shape:", X_test_pca.shape)

## 9. Đánh giá PCA

In [ ]:
pc1_variance = pca.explained_variance_ratio_[0]
pc2_variance = pca.explained_variance_ratio_[1]
total_explained_variance = pca.explained_variance_ratio_.sum()

original_features = X_train_scaled.shape[1]
pca_components = X_train_pca.shape[1]

dimensionality_reduction_ratio = (1 - pca_components / original_features) * 100
information_loss = (1 - total_explained_variance) * 100

print("\n===== PCA EVALUATION METRICS =====")
print("Original number of features:", original_features)
print("Number of PCA components:", pca_components)

print("\nExplained Variance Ratio:")
print("PC1 explained variance:", pc1_variance)
print("PC2 explained variance:", pc2_variance)
print("Total explained variance retained:", total_explained_variance)

print("\nDimensionality reduction ratio:", dimensionality_reduction_ratio, "%")
print("Information loss:", information_loss, "%")

## 10. Reconstruction error

In [ ]:
X_train_reconstructed = pca.inverse_transform(X_train_pca)
X_test_reconstructed = pca.inverse_transform(X_test_pca)

train_reconstruction_error = np.mean((X_train_scaled - X_train_reconstructed) ** 2)
test_reconstruction_error = np.mean((X_test_scaled - X_test_reconstructed) ** 2)

print("\n===== RECONSTRUCTION ERROR =====")
print("Training reconstruction error:", train_reconstruction_error)
print("Testing reconstruction error:", test_reconstruction_error)

## 11. Tạo dataframe kết quả PCA

In [ ]:
train_pca_df = pd.DataFrame(X_train_pca, columns=["PC1", "PC2"])
train_pca_df["class"] = y_train.values
train_pca_df["set"] = "train"

test_pca_df = pd.DataFrame(X_test_pca, columns=["PC1", "PC2"])
test_pca_df["class"] = y_test.values
test_pca_df["set"] = "test"

pca_result_df = pd.concat([train_pca_df, test_pca_df], ignore_index=True)

print("\n===== PCA OUTPUT DATA =====")
print(pca_result_df.head())

## 12. Trực quan hóa PCA

In [ ]:
plt.figure(figsize=(8, 6))

for wine_class in sorted(pca_result_df["class"].unique()):
    subset = pca_result_df[pca_result_df["class"] == wine_class]
    plt.scatter(subset["PC1"], subset["PC2"], label=f"Class {wine_class}", alpha=0.8)

plt.title("PCA Visualization of Wine Dataset")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.grid(True)
plt.show()

## 13. Scree plot và cumulative variance

In [ ]:
pca_full = PCA()
pca_full.fit(X_train_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("\n===== EXPLAINED VARIANCE FOR ALL COMPONENTS =====")
for i, variance in enumerate(explained_variance):
    print(f"PC{i + 1}: {variance:.4f}")

print("\n===== CUMULATIVE EXPLAINED VARIANCE =====")
for i, variance in enumerate(cumulative_variance):
    print(f"{i + 1} components: {variance:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker="o")
plt.title("Scree Plot")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
plt.axhline(y=0.95, linestyle="--", label="95% variance threshold")
plt.title("Cumulative Explained Variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.legend()
plt.grid(True)
plt.show()

## 14. Tuning PCA

In [ ]:
pca_95 = PCA(n_components=0.95)

X_train_pca_95 = pca_95.fit_transform(X_train_scaled)
X_test_pca_95 = pca_95.transform(X_test_scaled)

print("\n===== PCA HYPERPARAMETER TUNING =====")
print("Hyperparameter: n_components")
print("Number of components needed to retain 95% variance:", pca_95.n_components_)
print("Total variance retained:", np.sum(pca_95.explained_variance_ratio_))

## 15. Classifier dùng dữ liệu PCA

In [ ]:
pca_classifier = LogisticRegression(max_iter=1000, random_state=42)
pca_classifier.fit(X_train_pca, y_train)

y_pred_pca = pca_classifier.predict(X_test_pca)
pca_accuracy = accuracy_score(y_test, y_pred_pca)

print("\n===== PCA CLASSIFIER RESULT =====")
print("Model: Logistic Regression")
print("Input: 2 PCA components")
print("PCA-based classifier accuracy:", pca_accuracy)

print("\nClassification report:")
print(classification_report(y_test, y_pred_pca))

## 16. So sánh cuối cùng

In [ ]:
print("\n===== FINAL COMPARISON =====")
print("Baseline accuracy using original 13 features:", baseline_accuracy)
print("Accuracy using 2 PCA components:", pca_accuracy)
print("Accuracy difference:", baseline_accuracy - pca_accuracy)

if pca_accuracy > baseline_accuracy:
    print("Result: PCA-reduced data exceeds the baseline accuracy.")
elif pca_accuracy == baseline_accuracy:
    print("Result: PCA-reduced data achieves the same accuracy as the baseline.")
else:
    print("Result: PCA-reduced data does not exceed the baseline accuracy.")
    print("However, this can still be acceptable because PCA reduces dimensionality from 13 features to 2 components.")

## 17. Lưu kết quả

In [ ]:
pca_result_df.to_csv("pca_wine_output.csv", index=False)

print("\n===== SAVED OUTPUT =====")
print("PCA result has been saved to 'pca_wine_output.csv'.")